# Phase 0: Project Setup, Config Loader & Architecture Decisions

Welcome to the interactive companion notebook for the **Hinglish Query Understanding Engine**.

### Problem Overview
Users on Indian real estate platforms type messy Hinglish queries like:
- `"noida me 2bhk sasta near metro"`
- `"gurgaon 3 bhk ready to move 1.5 cr tak"`
- `"furnished 1bhk rent dwarka 25k"`

Our goal is to parse these queries into validated JSON search filters at minimal cost and latency using a **cost-aware model cascade** (cheap model first, escalating to a large LLM only when confidence is low or schema validation fails).

## 1. Path Anchoring & Working Directory Discipline

**Why this matters in an interview:**
- In Python, relative file paths like `open("configs/prices.yaml")` depend on `os.getcwd()` (where the user runs the terminal command from).
- If someone runs `pytest` from the root, from `tests/`, or inside a Docker container, relative paths fail with `FileNotFoundError`.
- Solution: Always anchor paths relative to the file's known location (`PROJECT_ROOT = Path(__file__).resolve().parent.parent`).

In [ ]:
import os
from pathlib import Path

# Locate project root relative to this notebook:
current_dir = Path().resolve()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
configs_dir = project_root / "configs"

print(f"Working Directory : {os.getcwd()}")
print(f"Project Root      : {project_root}")
print(f"Configs Directory : {configs_dir}")

# Verify configs directory exists
assert configs_dir.exists(), "configs directory must exist!"
print("Path check passed!")

## 2. Dated Model Pricing (`configs/prices.yaml`)

**Why externalize pricing?**
- API prices change over time.
- Hardcoding prices in code causes evaluation drift and invalidates published cost numbers.
- By pinning prices to a specific date (`2024-10-01`), all evaluations in `eval/metrics.py` compute reproducible cost-per-1k-queries.

In [ ]:
import yaml

prices_file = configs_dir / "prices.yaml"
with open(prices_file, "r", encoding="utf-8") as f:
    prices_data = yaml.safe_load(f)

print(f"Pricing Effective Date : {prices_data['effective_date']}")
print(f"Currency               : {prices_data['currency']}")
print("\nConfigured Models & Costs (per 1M tokens):")
for model_name, info in prices_data["models"].items():
    print(f"  - {model_name:<18}: Input=${info['input_cost_per_1m']:<5.3f} | Output=${info['output_cost_per_1m']:<5.3f}")

## 3. Safe Configuration Loader (`src/config_loader.py`)

Let's inspect the `load_yaml` implementation line by line.

**Key guards:**
1. `path.exists()` check -> gives clear `FileNotFoundError`.
2. `data is None` check -> catches accidentally empty YAML files early.
3. `isinstance(data, dict)` check -> prevents list-based or scalar YAML files from causing downstream `TypeError`.

In [ ]:
from typing import Any, Dict

def load_yaml(file_path: Path | str) -> Dict[str, Any]:
    """Safely load and validate a YAML configuration file."""
    path = Path(file_path)
    
    # If relative path, anchor to project root
    if not path.is_absolute():
        path = project_root / path
        
    # Guard 1: File existence
    if not path.exists():
        raise FileNotFoundError(f"Configuration file not found at: {path}")
        
    # Open and parse
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
        
    # Guard 2: Reject empty files
    if data is None:
        raise ValueError(f"Configuration file is empty: {path}")
        
    # Guard 3: Reject non-mapping roots (e.g. lists)
    if not isinstance(data, dict):
        raise ValueError(f"Configuration file root must be a mapping/dict: {path}")
        
    return data

# Test on prices.yaml
loaded_config = load_yaml("configs/prices.yaml")
print("Successfully loaded config via load_yaml()!")
print(f"Models loaded: {list(loaded_config['models'].keys())}")

## 4. Edge Cases & Exception Handling

In interviews, writing resilient code means proving how your code handles failure modes.
Let's test all 3 failure conditions interactively.

In [ ]:
import tempfile

# Test 1: Non-existent file
try:
    load_yaml("configs/non_existent.yaml")
except FileNotFoundError as e:
    print(f"[OK] Caught expected FileNotFoundError: {e}")

# Test 2: Completely empty YAML file
with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w", delete=False) as tmp:
    tmp_empty = Path(tmp.name)
try:
    load_yaml(tmp_empty)
except ValueError as e:
    print(f"[OK] Caught expected ValueError on empty file: {e}")
finally:
    tmp_empty.unlink()

# Test 3: List-based YAML file (invalid root)
with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w", delete=False) as tmp:
    tmp.write("- item1\n- item2\n")
    tmp_list = Path(tmp.name)
try:
    load_yaml(tmp_list)
except ValueError as e:
    print(f"[OK] Caught expected ValueError on list root: {e}")
finally:
    tmp_list.unlink()

## 5. Architectural Record & Pre-Registered Target

### Pre-Registered Target
> **"The cascade's field-level F1 is within 3 points of the large-LLM baseline (B3), judged by a paired bootstrap on the F1 difference where the lower bound of the 95% CI is above -3, at under 25% of B3's cost per 1k queries."**

### Key Principles for Interview Discussion:
1. **Why Paired Bootstrap?**
   Comparing raw mean F1 scores (e.g. 88.5% vs 89.2%) is statistically misleading on small test sets (~140 queries). Paired bootstrap evaluates the exact difference $\Delta F_1 = F_{1,\text{cascade}} - F_{1,\text{B3}}$ across resamples to establish true statistical significance.
2. **Data Split Discipline:**
   - **DEV (~60 queries):** Used for few-shot selection, prompt engineering, threshold tuning, and confidence calibration.
   - **TEST (~140 queries):** STRICTLY LOCKED. Scored only once at the end. Never used for tuning.
3. **Domain Agnosticism:**
   All entity extraction rules, unit conversions (k, lakh, cr), and aliases live in `configs/`, keeping `src/` reusable for any domain (e.g. jobs).